<a href="https://colab.research.google.com/github/vignesh-potharaj/StudyBot/blob/main/studybot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#Cell 1 — install
!pip install transformers torch --quiet


In [6]:
# Cell 2 — imports

from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

In [7]:
# Cell 3 — load tokeniser and model
# output_attentions=True tells the model to return attention tensors
# instead of computing them internally and discarding them
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval() # inference mode: no dropout, no gradient tracking
print("Model loaded:", MODEL_NAME)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: distilbert-base-uncased


In [8]:
# Cell 4 — tokenise the study question
# return_tensors="pt" = PyTorch tensors
# add_special_tokens=True adds [CLS] at start, [SEP] at end
question = "Explain Newton's third law of motion."
inputs = tokenizer(question, return_tensors="pt", add_special_tokens=True)
token_ids = inputs["input_ids"][0]
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print("Tokens:", tokens)


Tokens: ['[CLS]', 'explain', 'newton', "'", 's', 'third', 'law', 'of', 'motion', '.', '[SEP]']


In [9]:
# Cell 5 — forward pass
# torch.no_grad() disables gradient computation (saves memory during inference)
# outputs.attentions is a tuple of 6 tensors, one per transformer layer
with torch.no_grad():
 outputs = model(**inputs)
print("Number of attention layers:", len(outputs.attentions)) # 6
print("Layer 0 shape:", outputs.attentions[0].shape)
# torch.Size([1, 12, 11, 11])
# (batch=1, heads=12, from_token=11, to_token=11)


Number of attention layers: 6
Layer 0 shape: torch.Size([1, 12, 11, 11])


In [10]:
def top_attention_pairs(model_outputs, tokens, layer, head, k=5):
    """
    Safely print the top-k attention pairs by explicitly passing dependencies.
    """
    # Ensure attentions exist
    if model_outputs.attentions is None:
        raise ValueError("Model was not loaded with output_attentions=True")

    attn = model_outputs.attentions[layer][0][head].numpy()
    flat = attn.flatten()
    top_idx = flat.argsort()[::-1][:k]

    # Get the matrix dimensions explicitly
    num_rows, num_cols = attn.shape

    print(f"\nLayer {layer}, Head {head}")
    print(f"{'FROM':<14} {'TO':<14} {'WEIGHT':>8}")
    print("-" * 40)
    for idx in top_idx:
        row = idx // num_cols
        col = idx % num_cols
        print(f"{tokens[row]:<14} {tokens[col]:<14} {flat[idx]:.3f}")

# Call it by passing the actual variables:
top_attention_pairs(outputs, tokens, layer=0, head=0, k=5)


Layer 0, Head 0
FROM           TO               WEIGHT
----------------------------------------
newton         motion         0.308
s              [CLS]          0.300
of             [CLS]          0.273
[SEP]          [SEP]          0.266
'              [CLS]          0.265
